# Train a ResNet-101 COCO attribute predictor

This follows the paper's attribute path: build 1,000 image labels from COCO captions, fine-tune ImageNet-pretrained ResNet-101 with multi-label BCE, then predict five attributes for each MMSD image. COCO and MMSD image files must already be available locally. The paper does not specify tokenization, frequency cutoff, or training schedule; choices below are explicit reproduction settings.

Outputs are a checkpoint, attribute list, and `extract_all`-style rows (`[tweet_id, word1, ..., word5]`). This does not create the paper's Twitter-trained GloVe attribute vectors.

In [ ]:
import json, re, random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm

COCO_IMAGES = Path('/kaggle/input/coco-2017/train2017')
COCO_CAPTIONS = Path('/kaggle/input/coco-2017/annotations/captions_train2017.json')
MMSD_IMAGES = Path('/kaggle/input/datasets/fruich/mmsd-full/data/raw/dataset_image')
OUTPUT_DIR = Path('/kaggle/working/features/resnet101_coco_attributes')
BATCH_SIZE, EPOCHS, SEED = 64, 8, 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert COCO_IMAGES.is_dir() and COCO_CAPTIONS.is_file() and MMSD_IMAGES.is_dir()
print('Device:', DEVICE)

## Build image-level labels from COCO captions

For each image, use the union of words appearing in its captions. The top 1,000 corpus words become multi-label targets. This is a practical approximation of the paper's described caption-derived attributes; it may not match the authors' exact preprocessing.

In [ ]:
def words(text):
    return re.findall(r"[a-z]+", text.lower())

coco = json.loads(COCO_CAPTIONS.read_text())
captions_by_id = defaultdict(list)
for ann in coco['annotations']:
    captions_by_id[ann['image_id']].append(ann['caption'])

counts = Counter(token for captions in captions_by_id.values() for caption in captions for token in set(words(caption)))
attributes = [word for word, _ in counts.most_common(1000)]
assert len(attributes) == 1000
attribute_to_id = {word: i for i, word in enumerate(attributes)}
image_rows = [(int(image_id), [attribute_to_id[w] for w in {token for c in captions for token in words(c)} if w in attribute_to_id])
              for image_id, captions in captions_by_id.items()]
print('COCO images with captions:', len(image_rows), '| attributes:', len(attributes))

In [ ]:
# COCO official train/validation image split, keyed by image IDs in the caption annotations.
image_ids = sorted(captions_by_id)
random.Random(SEED).shuffle(image_ids)
split_at = int(0.9 * len(image_ids))
train_ids, valid_ids = set(image_ids[:split_at]), set(image_ids[split_at:])
labels_by_id = dict(image_rows)
transform = transforms.Compose([transforms.Resize((448, 448)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class CocoAttributes(Dataset):
    def __init__(self, ids): self.ids = sorted(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        image_id = self.ids[i]
        path = COCO_IMAGES / f'{image_id:012d}.jpg'
        with Image.open(path) as im: image = transform(im.convert('RGB'))
        target = torch.zeros(1000)
        target[labels_by_id[image_id]] = 1
        return image, target

train_loader = DataLoader(CocoAttributes(train_ids), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=DEVICE.type == 'cuda')
valid_loader = DataLoader(CocoAttributes(valid_ids), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=DEVICE.type == 'cuda')
model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 1000)
model = model.to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
print('GPUs:', torch.cuda.device_count())
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_loss = float('inf')
for epoch in range(EPOCHS):
    model.train(); train_loss = 0.0
    for images, targets in tqdm(train_loader, desc=f'Train {epoch+1}/{EPOCHS}'):
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), targets)
        loss.backward(); optimizer.step(); train_loss += loss.item() * len(images)
    model.eval(); valid_loss = 0.0
    with torch.inference_mode():
        for images, targets in valid_loader:
            valid_loss += criterion(model(images.to(DEVICE)), targets.to(DEVICE)).item() * len(images)
    valid_loss /= len(valid_ids)
    print({'epoch': epoch+1, 'train_loss': train_loss/len(train_ids), 'validation_loss': valid_loss})
    if valid_loss < best_loss:
        best_loss = valid_loss
        torch.save({'model': model.state_dict(), 'attributes': attributes, 'validation_loss': best_loss}, OUTPUT_DIR / 'resnet101_coco.pt')
(OUTPUT_DIR / 'attributes.json').write_text(json.dumps(attributes, indent=2) + '\n')

## Predict top-five attributes for MMSD images

Writes rows compatible with the repository's `extract/extract_all` reader. Predictions cover every `.jpg` in `MMSD_IMAGES`, matching the feature-extraction notebook's image enumeration.

In [ ]:
checkpoint = torch.load(OUTPUT_DIR / 'resnet101_coco.pt', map_location='cpu', weights_only=True)
attributes = checkpoint['attributes']
model.load_state_dict(checkpoint['model']); model.eval().to(DEVICE)
image_paths = sorted(MMSD_IMAGES.glob('*.jpg'), key=lambda p: p.stem)
rows = []
with torch.inference_mode():
    for start in tqdm(range(0, len(image_paths), BATCH_SIZE), desc='Predict MMSD attributes'):
        paths = image_paths[start:start+BATCH_SIZE]
        batch = []
        for path in paths:
            with Image.open(path) as im: batch.append(transform(im.convert('RGB')))
        probs = model(torch.stack(batch).to(DEVICE)).sigmoid().cpu()
        for path, scores in zip(paths, probs):
            top = scores.topk(5).indices.tolist()
            rows.append([path.stem, *[attributes[i] for i in top]])

out = OUTPUT_DIR / 'extract_all'
out.write_text(''.join(repr(row) + '\n' for row in rows))
print('Wrote', len(rows), 'rows to', out)